# Figure 4: image-level UMAP panels

Generates the UMAP panels of Figure 4 from the image-level UMAP table and the likelihood-model predicted timepoints.

**Inputs**
- `data/umap_table.csv`: one row per image (2,285 images): UMAP coordinates, compound and dose. Built from the per-model scores in `../Jill_analysis/` by `01_build_umap_table.py`.
- `../product/expected_timepoint_all20260306.tsv`: predicted timepoint per compound and dose (output of the likelihood model, `../DENV_TimeOfArrest_v2.qmd`).

**Outputs** (`figures/`): SVG files for each panel.

## 1. Imports and paths

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import matplotlib.cm as cm
import pandas as pd
from pathlib import Path

In [ ]:
import matplotlib.gridspec as gridspec

FIGSIZE = (8, 9)
AX_RECT = [0.12, 0.22, 0.8, 0.68]  
                                     # plot box

In [ ]:
# Paths are relative to this notebook (umap_analysis/) inside the repository.
DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

UMAP_TABLE = DATA_DIR / "umap_table.csv"                                  # image-level UMAP + plate metadata
EXPECTED_TIMEPOINT = Path("..") / "product" / "expected_timepoint_all20260306.tsv"  # likelihood-model output

In [ ]:
df_umap = pd.read_csv(UMAP_TABLE)

print(df_umap.shape)
df_umap.head()

## 2. UMAP colored by predicted timepoint

In [ ]:
required_cols = ["UMAP_1", "UMAP_2", "Compound", "Concentration(uM)"]

for col in required_cols:
    print(col, "OK" if col in df_umap.columns else "MISSING")

print("\nMissing values:")
print(df_umap[required_cols].isnull().sum())

In [ ]:
df_tp = pd.read_csv(EXPECTED_TIMEPOINT, sep="	")

df_umap = df_umap.rename(columns={
    "Compound": "treatment",
    "Concentration(uM)": "dose_uM"
})

df_umap["treatment"] = df_umap["treatment"].astype(str).str.strip()
df_tp["treatment"] = df_tp["treatment"].astype(str).str.strip()

# Harmonize compound spellings with the likelihood-model file (hyphen / space / case differences)
TREATMENT_NAMES = {
    "IMD-0354": "IMD 0354",
    "Remdesivir": "remdesivir",
    "SB-271046": "SB 271046",
}
df_umap["treatment"] = df_umap["treatment"].replace(TREATMENT_NAMES)

df_umap["dose_uM"] = pd.to_numeric(df_umap["dose_uM"], errors="coerce")
df_tp["dose_uM"] = pd.to_numeric(df_tp["dose_uM"], errors="coerce")

df = df_umap.merge(
    df_tp[["treatment", "dose_uM", "expected_timepoint"]],
    on=["treatment", "dose_uM"],
    how="left"
)

print("Merged shape:", df.shape)
print("Missing mappings:", df["expected_timepoint"].isna().sum())
assert df["expected_timepoint"].isna().sum() == 0, "some images have no predicted timepoint"

uniq_tp = sorted(df["expected_timepoint"].dropna().unique())

uniq_tp = sorted(df["expected_timepoint"].dropna().unique())

base_cmap = cm.get_cmap("plasma")

truncated_plasma = LinearSegmentedColormap.from_list(
    "truncated_plasma",
    base_cmap(np.linspace(0, 0.85, 256))
)

fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_axes(AX_RECT)

sc = ax.scatter(
    df["UMAP_1"],
    df["UMAP_2"],
    c=df["expected_timepoint"],
    cmap=truncated_plasma,
    vmin=0,
    vmax=48,
    s=15,
    alpha=0.9,
    linewidths=0
)

# colorbar lives in its own fixed strip below the axes, not stealing space from ax
cax = fig.add_axes([0.12, 0.10, 0.8, 0.03])  # [left, bottom, width, height]
cbar = fig.colorbar(sc, cax=cax, orientation="horizontal")
cbar.set_label("Predicted Timepoint (hpi)", fontsize=12, labelpad=8)
cbar.set_ticks([0, 12, 24, 36, 48])

ax.grid(False)
ax.set_title(
    "Image-Level UMAP Colored by Representative Predicted Timepoint",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("UMAP 1", fontsize=12)
ax.set_ylabel("UMAP 2", fontsize=12)

plt.savefig(
    FIG_DIR / "figure4_umap_representative_timepoint_scaled.svg",
    format="svg"
)  # no bbox_inches="tight" here — keeps the fixed figure geometry intact

plt.show()

## 3. LY411575 dose UMAP

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

FIGSIZE = (8, 9)          # identical to Figure 1
AX_RECT = [0.12, 0.22, 0.8, 0.68]   # identical plot box as Figure 1

# --- data prep ---
ly_df = df[df["treatment"] == "LY411575"].copy()

ly_df["dose_uM"] = pd.to_numeric(
    ly_df["dose_uM"],
    errors="coerce"
)

dose_levels = sorted(
    ly_df["dose_uM"].dropna().unique()
)

base_cmap = cm.get_cmap("plasma")

color_positions = np.linspace(0, 0.85, len(dose_levels))
dose_colors = [base_cmap(p) for p in color_positions]

dose_cmap = dict(zip(dose_levels, dose_colors))

# --- discrete colormap setup for the legend strip ---
disc_cmap = ListedColormap(dose_colors)
bounds = np.arange(len(dose_levels) + 1)
norm = BoundaryNorm(bounds, disc_cmap.N)

# --- figure ---
fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_axes(AX_RECT)

sns.scatterplot(
    data=df,
    x="UMAP_1",
    y="UMAP_2",
    color="lightgrey",
    s=12,
    alpha=0.5,
    linewidth=0,
    legend=False,
    ax=ax
)

sns.scatterplot(
    data=ly_df,
    x="UMAP_1",
    y="UMAP_2",
    hue="dose_uM",
    hue_order=dose_levels,
    palette=dose_cmap,
    s=15,
    alpha=0.9,
    linewidth=0,
    legend=False,
    ax=ax
)

cax = fig.add_axes([0.12, 0.10, 0.8, 0.03])
cb = fig.colorbar(
    cm.ScalarMappable(norm=norm, cmap=disc_cmap),
    cax=cax,
    orientation="horizontal",
    boundaries=bounds,
    ticks=np.arange(len(dose_levels)) + 0.5
)
cb.ax.set_xticklabels([f"{d:g}" for d in dose_levels])
cb.set_label(r"LY411575 Dose ($\mu$M)", fontsize=12, labelpad=8)

ax.grid(False)

ax.set_title(
    "Image-Level UMAP Colored by LY411575 Dose Distribution",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("UMAP 1", fontsize=12)
ax.set_ylabel("UMAP 2", fontsize=12)

plt.savefig(
    FIG_DIR / "figure4_umap_ly411575_dose_colorbar.svg",
    format="svg"
)

plt.show()

## 4. LY411575 and pralatrexate

In [ ]:
from matplotlib.patches import Patch

# Select only the two treatments to highlight
plot_df = df[df["treatment"].isin(["LY411575", "Pralatrexate"])].copy()

# Colors for each treatment
treatment_colors = {
    "LY411575": "red",
    "Pralatrexate": "blue"
}

fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_axes(AX_RECT)

# Plot all images in the background
sns.scatterplot(
    data=df,
    x="UMAP_1",
    y="UMAP_2",
    color="lightgrey",
    s=12,
    alpha=0.5,
    linewidth=0,
    legend=False,
    ax=ax
)

# Overlay the two treatments
sns.scatterplot(
    data=plot_df,
    x="UMAP_1",
    y="UMAP_2",
    hue="treatment",
    hue_order=["LY411575", "Pralatrexate"],
    palette=treatment_colors,
    s=15,
    alpha=0.9,
    linewidth=0,
    legend=False,
    ax=ax
)

# Custom legend
handles = [
    Patch(color="red", label="LY411575"),
    Patch(color="blue", label="Pralatrexate")
]

ax.legend(
    handles=handles,
    title="Treatment",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    ncol=2,
    fontsize=12,
    title_fontsize=14,
    frameon=False,
    handlelength=1.2,
    handletextpad=0.5,
    columnspacing=1.0
)

ax.grid(False)
ax.set_title(
    "Image-Level UMAP Highlighting LY411575 and Pralatrexate",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("UMAP 1", fontsize=12)
ax.set_ylabel("UMAP 2", fontsize=12)

plt.savefig(
    FIG_DIR / "figure4_umap_ly411575_pralatrexate.svg",
    format="svg"
)

plt.show()